# dspk on a Neuropixels recording

[dsp-kitchen](https://github.com/neuro-kitchen/dsp-kitchen) processes multichannel recordings on
any GPU, or on the CPU, from one code base. This notebook installs it, opens a public
Neuropixels 1.0 recording (385 channels, 30 kHz, 45 s), cleans it on the GPU, sorts it with
dspk's Kilosort4 and looks at the units.

**Runtime → Change runtime type → T4 GPU** makes it fast; without a GPU it still runs, on the CPU.

## 1. Install

In [ ]:
%pip install -q "dspk[synapse]" matplotlib

In [ ]:
import dspk

print("dspk", dspk.__version__)
print("runtimes compiled in:", dspk.runtime.available())

dspk has four runtimes: `cuda` (NVIDIA), `wgpu` (any GPU through Vulkan, Metal or DirectX),
`hip` (AMD) and `cpu`. The cell below picks the first one that opens a device here.

In [ ]:
import numpy as np


def first_working_runtime():
    probe = np.zeros((2, 64), dtype=np.float32)
    for name in ("cuda", "wgpu", "cpu"):
        try:
            dspk.ops.spatial.car(probe, runtime=name)
            return name
        except Exception as error:  # no device or driver for this runtime
            print(f"{name}: not here ({str(error)[:60]}…)")
    raise RuntimeError("no runtime could open a device")


RUNTIME = first_working_runtime()
dspk.runtime.set(RUNTIME)
print("using:", RUNTIME)

## 2. The recording

Kilosort4's test recording (Pachitariu et al.), from the address its documentation gives:
1 GB, a zip of one SpikeGLX `.bin`. dspk reads a raw binary with a small JSON file next to it
that says how it is laid out.

In [ ]:
import hashlib
import json
import shutil
import urllib.request
import zipfile
from pathlib import Path

URL = "https://www.kilosort.org/downloads/ZFM-02370_mini.imec0.ap.short.zip"
NAME = "ZFM-02370_mini.imec0.ap.short.bin"
SHA256 = "918be4edccddbb1a0d420afae649cf0f33f2f6720365d743fa9cb0923da7b2b9"
DATA = Path("data")
DATA.mkdir(exist_ok=True)
recording_path = DATA / NAME

if not recording_path.exists():
    archive = DATA / "download.zip"
    print("downloading", URL, "…")
    urllib.request.urlretrieve(URL, archive)
    with zipfile.ZipFile(archive) as z:
        member = next(m for m in z.namelist() if m.endswith(NAME))
        with z.open(member) as src, recording_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    archive.unlink()

digest = hashlib.sha256()
with recording_path.open("rb") as f:
    while chunk := f.read(1 << 24):
        digest.update(chunk)
assert digest.hexdigest() == SHA256, "the download is not the file this notebook was written with"

# SpikeGLX imec AP band: int16, 2.34375 µV per step
meta = {"channels": 385, "sample_rate_hz": 30000.0, "format": "int16", "order": "time_major", "gain_uv": 2.34375, "offset_uv": 0.0, "header_bytes": 0, "samples": 1350000}
recording_path.with_suffix(".meta").write_text(json.dumps(meta, indent=2))

recording = dspk.io.open(recording_path)
print(recording)

## 3. Clean one second on the GPU

Two steps every spike-sorting pipeline starts with: a band-pass (300–6000 Hz) to keep the spike
band, then a common average reference (the mean across channels, subtracted at each sample) to
remove what every channel shares. Each call runs on the device and returns a NumPy array.

In [ ]:
import time

fs = recording.sample_rate
start = int(10 * fs)
raw = recording.read(start=start, stop=start + int(fs))  # (channels, samples)

t0 = time.perf_counter()
filtered = dspk.ops.filter.iir(raw, fs, "bandpass", (300.0, 6000.0))
clean = dspk.ops.spatial.car(filtered)
print(f"{raw.shape[0]} channels × {raw.shape[1]} samples cleaned in {(time.perf_counter() - t0) * 1e3:.0f} ms on {RUNTIME} (the first call also compiles the kernels)")

t0 = time.perf_counter()
clean = dspk.ops.spatial.car(dspk.ops.filter.iir(raw, fs, "bandpass", (300.0, 6000.0)))
print(f"again: {(time.perf_counter() - t0) * 1e3:.0f} ms")

In [ ]:
import matplotlib.pyplot as plt

channels = np.arange(180, 200)
window = slice(0, int(0.1 * fs))
t_ms = np.arange(window.stop) / fs * 1e3
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, data, title in ((axes[0], raw, "raw"), (axes[1], clean, "band-pass 300–6000 Hz + common average reference")):
    block = np.asarray(data[channels, window], dtype=np.float32)
    block = block - block.mean(axis=1, keepdims=True)
    step = 6 * np.median(np.abs(block)) / 0.6745 + 1e-9
    for row, trace in enumerate(block):
        ax.plot(t_ms, trace / step + row, lw=0.6, color="k")
    ax.set_title(title)
    ax.set_xlabel("time (ms)")
axes[0].set_ylabel("channel (180–199)")
plt.tight_layout()
plt.show()

## 4. Sort it with Kilosort4

The sorters are a separate program, `dspk-cli-sorter` (GPL-3.0, because Kilosort4 is):
`dspk sorters install` downloads the build of this version and checks it against the release's
manifest. It writes a `.sorting.zarr`, dspk's format for sortings.

In [ ]:
import subprocess
import sys

dspk_command = shutil.which("dspk") or str(Path(sys.executable).parent / "dspk")
subprocess.run([dspk_command, "sorters", "install"], check=True)
status = subprocess.run([dspk_command, "sorters", "status"], check=True, capture_output=True, text=True).stdout
sorter = status.rsplit(" (", 1)[0].strip()
print(sorter)

In [ ]:
sorting_path = DATA / "kilosort4.sorting.zarr"
t0 = time.perf_counter()
subprocess.run([sorter, "--runtime", RUNTIME, "kilosort4", str(recording_path), "--probe", "neuropixels1", "--output", str(sorting_path)], check=True)
print(f"sorted {recording.duration:.0f} s of {recording.channel_count} channels in {time.perf_counter() - t0:.0f} s on {RUNTIME}")

## 5. The units

In [ ]:
from dspk.synapse import io, metrics

sorting = io.Sorting.load(sorting_path)
print(sorting)

units = sorting.units()
rate = np.asarray(units["firing_rate_hz"])
isi = np.asarray(units["isi_violation_ratio"])
single = np.asarray(units["label"]) == "SingleUnit"
print(f"{len(rate)} units, {single.sum()} labelled single units")

In [ ]:
ids = np.asarray(units["unit_id"])
busiest = int(ids[single][np.argmax(np.asarray(units["spikes"])[single])])
acg = metrics.autocorrelogram(sorting.spike_train(busiest), sorting.sample_rate, bin_ms=0.5, window_ms=50.0)
counts = np.asarray(acg["counts"])
lags = (np.arange(len(counts)) - len(counts) // 2) * 0.5

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].scatter(rate[~single], isi[~single] * 100, s=10, color="0.6", label="multi-unit")
axes[0].scatter(rate[single], isi[single] * 100, s=10, color="tab:green", label="single unit")
axes[0].set_xscale("log")
axes[0].set_xlabel("firing rate (Hz)")
axes[0].set_ylabel("ISI violations (% of expected)")
axes[0].legend()
axes[1].bar(lags, counts, width=0.5, color="tab:blue")
axes[1].set_xlabel("lag (ms)")
axes[1].set_title(f"unit {busiest}: autocorrelogram")
first_second = [(sorting.spike_train(int(u)) / sorting.sample_rate) for u in ids[single][:60]]
axes[2].eventplot([t[t < 5.0] for t in first_second], colors="k", linewidths=0.5)
axes[2].set_xlabel("time (s)")
axes[2].set_ylabel("single units (first 60)")
axes[2].set_title("first 5 s")
plt.tight_layout()
plt.show()

## Where next

- The desktop app (explore, pipelines, sorting, curation): `pip install "dspk[app]"`, then `dspk app`.
- The documentation: https://neuro-kitchen.github.io/dsp-kitchen/
- The source, and issues: https://github.com/neuro-kitchen/dsp-kitchen

The recording is Kilosort4's test data (Pachitariu, Sridhar, Pennington & Stringer, *Nature
Methods* 2024), downloaded from kilosort.org.